# Coordinating agents that don't share memory: a message-bus consensus + liveness pattern

In production, your agents rarely live in one Python process. A nightly worker runs on one box, an interactive assistant on another, a webhook handler in a third container. They don't share a call stack or memory. Two problems appear the moment more than one agent can touch the same irreversible action (send the customer email, charge the card, merge the PR): **who actually does it, and how do you guarantee it happens exactly once?** And when one agent crashes mid-task, how does the rest of the group notice and still make progress instead of hanging forever?

In-process orchestration patterns (orchestrator-workers, parallel voting) assume a shared process to coordinate through. Once your agents are distributed, that assumption breaks. This recipe gives you the smallest coordination layer that survives it: an append-only message bus as the *only* channel, a propose → vote → commit consensus so a shared action runs exactly once, and a heartbeat so a dead peer is detected and routed around.

**By the end of this cookbook, you'll be able to:**
- Coordinate independent Claude agents that share no memory, using an append-only message bus as their only channel
- Reach agreement on a shared irreversible action with a propose → vote → commit protocol so it executes exactly once
- Detect a crashed peer with heartbeats and fail over to a live one without double-executing the action
- Reason about when distributed consensus is worth it versus a single in-process orchestrator

This pattern extends to multi-session assistants that must not double-act, fan-out across hosts, and always-on background fleets that need to survive a node going dark.

## Prerequisites

Before following this guide, ensure you have:

**Required Knowledge:**
- Python fundamentals: functions, `async`/`await`, and basic data structures
- Comfort with the Anthropic Messages API (a single `client.messages.create` call)

**Required Tools:**
- Python 3.11 or higher
- Anthropic API key ([get one here](https://console.anthropic.com)). *Optional:* set `DEMO_MODE=true` to run the whole notebook with no key and no cost

**Recommended:**
- A feel for why "exactly-once" is hard once state is not shared (we keep the treatment light)

## Setup

First, install the required dependencies:

In [ ]:
%%capture
%pip install -U anthropic python-dotenv

In [ ]:
import asyncio
import hashlib
import json
import os
import sqlite3
import time
from dataclasses import dataclass

import anthropic
import dotenv

dotenv.load_dotenv()

# A constant model name is easier to change in one place.
MODEL = "claude-sonnet-4-6"

# DEMO_MODE lets the whole notebook run with no API key and no cost.
DEMO_MODE = os.environ.get("DEMO_MODE", "false").lower() == "true"

client = None if DEMO_MODE else anthropic.Anthropic()

## 1. The bus: the only thing peers share

The single rule of this recipe: **peers never share Python objects.** Their only channel is an append-only log. We model it with one SQLite table so it is durable and easy to inspect. In a real deployment this is a Telegram group, a Redis stream, an SQS queue, or a synced file. The API is deliberately tiny: `post` an event, `read` events since an offset.

In [ ]:
class Bus:
    """Append-only, totally-ordered event log. The ONLY channel between peers."""

    def __init__(self, path=":memory:"):
        self._db = sqlite3.connect(path, check_same_thread=False)
        self._db.execute(
            "CREATE TABLE IF NOT EXISTS events ("
            "  seq INTEGER PRIMARY KEY AUTOINCREMENT,"
            "  ts REAL, sender TEXT, kind TEXT, body TEXT)"
        )
        self._db.commit()

    def post(self, sender: str, kind: str, body: dict):
        self._db.execute(
            "INSERT INTO events (ts, sender, kind, body) VALUES (?, ?, ?, ?)",
            (time.time(), sender, kind, json.dumps(body)),
        )
        self._db.commit()

    def read(self, since_seq: int = 0):
        cur = self._db.execute(
            "SELECT seq, ts, sender, kind, body FROM events WHERE seq > ? ORDER BY seq",
            (since_seq,),
        )
        return [
            {"seq": s, "ts": t, "sender": snd, "kind": k, "body": json.loads(b)}
            for (s, t, snd, k, b) in cur.fetchall()
        ]

The four message kinds we use are `HEARTBEAT`, `PROPOSE`, `VOTE`, and `COMMIT`. Consensus is just discipline about how peers react to these four.

## 2. A peer is a Claude conversation with a private inbox

Each peer has an `id`, its own view of the bus (a private read `offset`, so it cannot peek at another peer's memory), and one job: when a task appears, decide whether to **propose** an action, and when it sees someone else's proposal, **vote** on it. The decision is made by Claude, so each peer reasons independently from its own vantage point.

In [ ]:
@dataclass
class Peer:
    id: str
    bus: Bus
    offset: int = 0  # private cursor into the bus; peers do not share this

    def _decide(self, proposal: dict) -> dict:
        """Ask Claude to vote ACCEPT / REJECT on a proposed shared action."""
        prompt = (
            "You are one node in a fleet of independent agents that share no memory. "
            "A peer proposed a shared, irreversible action. Vote ACCEPT only if the "
            "action is clearly correct and safe to run exactly once.\n\n"
            f"Proposed action: {json.dumps(proposal, indent=2)}\n\n"
            'Reply with JSON only: {"vote": "ACCEPT" | "REJECT", "reason": "<one line>"}'
        )
        if DEMO_MODE:
            # Deterministic stand-in so the notebook runs with no API key.
            return {"vote": "ACCEPT", "reason": "demo: proposal is well-formed"}
        msg = client.messages.create(
            model=MODEL,
            max_tokens=200,
            messages=[{"role": "user", "content": prompt}],
        )
        return json.loads(msg.content[0].text)

    def heartbeat(self):
        self.bus.post(self.id, "HEARTBEAT", {})

    def propose(self, action: dict):
        # Content hash = the identity of the action. Everyone commits the SAME hash once.
        h = hashlib.sha256(json.dumps(action, sort_keys=True).encode()).hexdigest()[:12]
        self.bus.post(self.id, "PROPOSE", {"hash": h, "action": action})
        return h

## 3. Consensus: agree before anyone acts

The protocol is four rules every peer runs on each new bus event:

1. See a `PROPOSE` you haven't voted on → run `_decide`, post a `VOTE`.
2. Count `VOTE`s for a hash. On a quorum of ACCEPTs, the action is *decided*.
3. Exactly one peer commits: the **lowest-id live peer** (a deterministic tie-break, no extra round trips). It posts `COMMIT` and runs the side effect.
4. Anyone who sees a `COMMIT` for a hash treats that action as done. **Idempotency by hash** means a late or duplicate committer is a no-op.

In [ ]:
def live_peers(bus: Bus, window: float = 2.0):
    """Peers whose heartbeat is fresh within `window` seconds."""
    now = time.time()
    seen = {}
    for e in bus.read(0):
        if e["kind"] == "HEARTBEAT":
            seen[e["sender"]] = e["ts"]
    return sorted(pid for pid, ts in seen.items() if now - ts <= window)


def committed_hashes(bus: Bus):
    return {e["body"]["hash"] for e in bus.read(0) if e["kind"] == "COMMIT"}


def accepted_hashes(bus: Bus, quorum: int):
    tally = {}
    for e in bus.read(0):
        if e["kind"] == "VOTE" and e["body"].get("vote") == "ACCEPT":
            tally[e["body"]["hash"]] = tally.get(e["body"]["hash"], 0) + 1
    return [h for h, n in tally.items() if n >= quorum]


async def run_peer(peer: Peer, quorum: int, side_effect, stop_after: float):
    voted, deadline = set(), time.time() + stop_after
    while time.time() < deadline:
        for e in peer.bus.read(peer.offset):
            peer.offset = e["seq"]
            if e["kind"] == "PROPOSE" and e["body"]["hash"] not in voted:
                voted.add(e["body"]["hash"])
                verdict = peer._decide(e["body"])
                peer.bus.post(peer.id, "VOTE", {"hash": e["body"]["hash"], **verdict})

        # Tally, then commit-if-I'm-the-committer, guarded by idempotency.
        for h in accepted_hashes(peer.bus, quorum):
            if h in committed_hashes(peer.bus):
                continue  # someone already did it: stand down
            committer = live_peers(peer.bus)[0]  # deterministic, failover-safe
            if peer.id == committer:
                peer.bus.post(peer.id, "COMMIT", {"hash": h})
                side_effect(h)  # the one, real, irreversible action

        peer.heartbeat()
        await asyncio.sleep(0.2)

**Why the committer is `live_peers()[0]` and not a fixed leader:** it is recomputed from *fresh heartbeats* on every tick. If the peer that would have committed is dead, it isn't in `live_peers()`, so the next one takes over automatically. The `COMMIT`-hash guard means a recovered or lagging peer can never double-fire the action.

## 4. Run it: three peers, one email, exactly once

In [ ]:
def make_side_effect(log):
    def _fire(h):
        log.append(h)  # in real life: send the email / charge / merge
        print(f"  ✅ ACTION EXECUTED for {h} (exactly once)")

    return _fire


async def demo():
    bus = Bus()
    peers = [Peer(f"node-{i}", bus) for i in range(3)]
    for p in peers:  # warm up liveness
        p.heartbeat()
    fired = []
    peers[1].propose(
        {"type": "send_email", "to": "customer@acme.com", "subject": "Your report is ready"}
    )
    await asyncio.gather(
        *[run_peer(p, quorum=2, side_effect=make_side_effect(fired), stop_after=3.0) for p in peers]
    )
    print(f"\nTotal executions: {len(fired)}  (expected: 1)")


await demo()

  ✅ ACTION EXECUTED for 933681b3776b (exactly once)

Total executions: 1  (expected: 1)


Three independent peers, one proposal, **one** execution. No peer could see another's memory; they agreed entirely through the bus.

## 5. The failure test: kill the committer

The whole point is surviving a dead peer. We start the proposal, then stop heartbeating `node-0` (the would-be committer) *before* it commits. `node-1` should notice `node-0` is stale and take over, still exactly once.

In [ ]:
async def demo_failover():
    bus = Bus()
    peers = [Peer(f"node-{i}", bus) for i in range(3)]
    fired = []

    # node-0 heartbeats once, then goes dark (simulated crash: never heartbeats again).
    peers[0].heartbeat()
    peers[2].propose({"type": "merge_pr", "repo": "acme/api", "number": 412})

    async def crashed_committer(p):  # posts nothing further; falls out of live_peers()
        await asyncio.sleep(3.0)

    await asyncio.gather(
        crashed_committer(peers[0]),
        run_peer(peers[1], 2, make_side_effect(fired), 3.0),
        run_peer(peers[2], 2, make_side_effect(fired), 3.0),
    )
    print(f"\nCommitter crashed. Total executions: {len(fired)}  (expected: 1)")


await demo_failover()

  ✅ ACTION EXECUTED for 7f93e326065c (exactly once)

Committer crashed. Total executions: 1  (expected: 1)


A single-orchestrator design would have hung here: the orchestrator was the crashed node. The group routes around it because "who commits" is a *function of current liveness*, not a fixed role. Because commit is guarded by the action hash, `node-0` coming back later cannot re-fire the merge.

## Conclusion

You built the smallest coordination layer that keeps a fleet of memory-isolated Claude agents correct:

- **Message bus as the only channel**, so the pattern is honest about distribution (sections 1-2).
- **Propose → vote → commit with hash idempotency**, so a shared irreversible action runs exactly once even though no peer can see another's memory (sections 3-4).
- **Heartbeat-driven committer selection**, so a crashed peer is detected and routed around with no double-execution (section 5).

**When to reach for this:** more than one agent can trigger the same external side effect, or your agents genuinely run in separate processes, hosts, or sessions. **When not to:** if a single orchestrator can hold all the sub-agents in one process, use [`orchestrator_workers.ipynb`](orchestrator_workers.ipynb). It is simpler and you do not need consensus.

**Where to go next:**
- Swap the SQLite `Bus` for your real transport (a queue, a stream, a synced file).
- Add a `COUNTER` message so a peer can propose a *modified* action instead of a flat reject.
- Persist the vote tally so a peer that restarts rejoins a decision already in flight.